In [29]:
import pandas as pd
import importlib
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import edits_with_volume_plot as ewv
import tensorflow
import tensorflow_hub as hub
import numpy as np
import yfinance as yf
from statsmodels.tsa.stattools import adfuller
from ipywidgets import interact
_ = importlib.reload(ewv)

"""A shortcut for loading files from data directory"""
DR = "data/datasets/"

### Loading the SP500 Data

In [3]:
sp500 = yf.download("^GSPC", start="2009-01-01")
sp500 = sp500[["Close"]]
sp500["mkt_return"] = np.log(sp500["Close"] / sp500["Close"].shift(1))
sp500.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,mkt_return
Ticker,^GSPC,
Date,,
2009-01-02,931.799988,NaN
2009-01-05,927.450012,-0.004679
2009-01-06,934.700012,0.007787
2009-01-07,906.650024,-0.030469
2009-01-08,909.729980,0.003391


### Loading the tesla stock data

In [5]:
stock = ewv.prepare_stock_df(pd.read_csv(f"{DR}STOCK_tesla.csv"))
stock["log_return"] = np.log(stock['Close'] / stock['Close'].shift(1))

### Loading Elon Musk wiki data

In [6]:
msk = pd.read_csv(f"{DR}WIKI_elon_musk.csv")
msk = ewv.prepare_wiki_df(msk)

agg = ewv.get_aggregated_wiki_stats(msk, "D")

In [59]:
agg.index = agg.index.tz_localize(None)
stock.index = stock.index.tz_localize(None)
sp500.index = sp500.index.tz_localize(None)
agg.index = agg.index.normalize()
stock.index = stock.index.normalize()
sp500.index = sp500.index.normalize()

df_combined = agg.join(stock, how='inner')
df_combined = df_combined.join(sp500["mkt_return"], how='inner')

df_combined["mkt_return_l1"] = df_combined["mkt_return"].shift(1)
df_combined["log_return_l1"] = df_combined["log_return"].shift(1)
df_combined["log_volume_l1"] = np.log(df_combined["Volume"].shift(1))
# df_combined["volume_l1"] = df_combined["Volume"].shift(1)
df_combined["unique_editors_l1"] = df_combined["unique_editors"].shift(1)
df_combined["total_edits_l1"] = df_combined["total_edits"].shift(1)
df_combined["reverts_l1"] = df_combined["reverts"].shift(1)
df_combined["edit_diff"] = df_combined["total_edits"] - df_combined["total_edits_l1"]
df_combined.dropna(inplace=True)


In [34]:
df_combined.head(2)

,reverts,unique_editors,total_edits,Date,Close,Volume,log_return,mkt_return,mkt_return_l1,log_return_l1,log_volume_l1,unique_editors_l1,total_edits_l1,reverts_l1
timestamp,,,,,,,,,,,,,,
2010-07-01,0,1,2,2010-07-01 04:00:00+00:00,1.464,123282000,-0.081723,-0.003246,-0.010164,-0.002515,19.367720,1.0,4.0,0.0
2010-07-02,0,0,0,2010-07-02 04:00:00+00:00,1.280,77097000,-0.134312,-0.004673,-0.003246,-0.081723,18.629985,1.0,2.0,0.0


In [66]:
def interval_regression(a):
    # Use a copy so you don't destroy your original data!
    dff = df_combined[df_combined.Date.dt.year >= a].copy()

    # Use the filtered data (dff) in the regression
    lm = smf.ols("log_return ~ mkt_return + mkt_return_l1 + log_return_l1 + log_volume_l1 + total_edits", data=dff).fit(cov_type="HC1")
    print(f"--- Results for Year >= {a} ---")
    # print(lm.f_test("total_edits = total_edits_l1 = 0"))
    print(lm.summary())

interact(interval_regression, a=(2010, 2026, 1))

interactive(children=(IntSlider(value=2018, description='a', max=2026, min=2010), Output()), _dom_classes=('wi…

<function __main__.interval_regression(a)>